# SWE-Gym-Plus trajectory data quality audit

Purpose: assess whether an existing action-level SFT export is structurally trustworthy enough for a future training run. Grain is one supervised assistant action; task-level counts must be reported separately. The notebook reads private files supplied through `SWE_GYM_PLUS_ALIGNED_DATA` and prints only aggregate checks—never prompts, code, patches, credentials, or raw paths.

Set `SWE_GYM_PLUS_ALIGNED_DATA` to the private experiment directory containing `data/actions.jsonl` and `data/trajectories.jsonl`. Optionally set `SWE_GYM_PLUS_TOKENIZER` to a local tokenizer directory, `SWE_GYM_PLUS_EVENT_ROOT` to a local mirror of source event artifacts, and `SWE_GYM_PLUS_MAX_NEW_TOKENS` to the inference generation limit.

In [ ]:
import hashlib
import json
import os
from collections import Counter, defaultdict
from statistics import median
from pathlib import Path

DATA_ROOT = Path(os.environ['SWE_GYM_PLUS_ALIGNED_DATA']).expanduser()
ACTION_FILE = DATA_ROOT / 'data' / 'actions.jsonl'
TRAJECTORY_FILE = DATA_ROOT / 'data' / 'trajectories.jsonl'
assert ACTION_FILE.is_file() and TRAJECTORY_FILE.is_file(), 'private trajectory files are missing'

def read_jsonl(path):
    rows, invalid, blank = [], 0, 0
    with path.open(encoding='utf-8') as stream:
        for line in stream:
            if not line.strip():
                blank += 1
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                invalid += 1
    return rows, invalid, blank

actions, invalid_actions, blank_actions = read_jsonl(ACTION_FILE)
trajectories, invalid_trajectories, blank_trajectories = read_jsonl(TRAJECTORY_FILE)
print({'action_rows': len(actions), 'trajectory_rows': len(trajectories),
       'invalid_json_lines': invalid_actions + invalid_trajectories,
       'blank_lines': blank_actions + blank_trajectories})

In [ ]:
# Required-field, type, grain, duplication, target-shape, and coverage checks.
required = {'task_id': str, 'run_id': str, 'action_index': int, 'messages': list,
            'source_evaluation_verdict': str, 'assistant_only_loss': bool,
            'next_action_only_loss': bool, 'prompt_policy': str}
missing_by_field = {key: sum(key not in row or row[key] is None for row in actions) for key in required}
wrong_types = {key: sum(key in row and row[key] is not None and (type(row[key]) is not typ if typ in (int, bool) else not isinstance(row[key], typ)) for row in actions) for key, typ in required.items()}
key_pairs = [(json.dumps(row.get('task_id'), sort_keys=True), json.dumps(row.get('action_index'), sort_keys=True)) for row in actions]
task_rows = defaultdict(list)
for row in actions:
    task_rows[json.dumps(row.get('task_id'), sort_keys=True)].append(row)
exact_rows = [json.dumps(row, sort_keys=True, ensure_ascii=False, separators=(',', ':')) for row in actions]
role_shapes = Counter(json.dumps([m.get('role') for m in row.get('messages', []) if isinstance(m, dict)], sort_keys=True) for row in actions)
empty_targets = 0
invalid_target_json = 0
done_targets = 0
target_shapes = Counter()
for row in actions:
    messages = row.get('messages') or []
    target = messages[-1].get('content') if messages and isinstance(messages[-1], dict) else None
    if not isinstance(target, str) or not target.strip():
        empty_targets += 1
        continue
    try:
        action = json.loads(target)
        if not isinstance(action, dict):
            invalid_target_json += 1
        else:
            target_shapes[','.join(sorted(action))] += 1
            done_targets += action.get('done') is True
    except json.JSONDecodeError:
        invalid_target_json += 1

noncontiguous_tasks = 0
for rows in task_rows.values():
    indices = sorted(row['action_index'] for row in rows if type(row.get('action_index')) is int)
    if indices != list(range(len(rows))):
        noncontiguous_tasks += 1

print({'missing_required_by_field': missing_by_field, 'wrong_types_by_field': wrong_types,
       'duplicate_task_action_keys': len(key_pairs) - len(set(key_pairs)),
       'exact_duplicate_rows': len(exact_rows) - len(set(exact_rows)),
       'distinct_tasks': len(task_rows),
       'actions_per_task_min_median_max': (min(map(len, task_rows.values())),
           median(map(len, task_rows.values())), max(map(len, task_rows.values()))) if task_rows else None,
       'tasks_with_noncontiguous_action_indices': noncontiguous_tasks,
       'empty_targets': empty_targets, 'invalid_target_json': invalid_target_json,
       'done_targets': done_targets,
       'target_key_shapes': dict(target_shapes), 'message_role_shapes': dict(role_shapes),
       'evaluation_verdicts': dict(Counter(json.dumps(row.get('source_evaluation_verdict'), sort_keys=True) for row in actions)),
       'prompt_policies': dict(Counter(json.dumps(row.get('prompt_policy'), sort_keys=True) for row in actions))})

In [ ]:
# Cross-check action rows against the trajectory index and expose evidence gaps.
trajectory_by_task = {}
duplicate_trajectory_tasks = 0
for row in trajectories:
    task_id = json.dumps(row.get('task_id'), sort_keys=True)
    if task_id in trajectory_by_task:
        duplicate_trajectory_tasks += 1
    trajectory_by_task[task_id] = row
action_task_ids = set(task_rows)
trajectory_task_ids = set(trajectory_by_task)
link_mismatches = 0
for row in actions:
    source = trajectory_by_task.get(json.dumps(row.get('task_id'), sort_keys=True))
    if (source is None or source.get('run_id') != row.get('run_id') or
            source.get('evaluation_verdict') != row.get('source_evaluation_verdict')):
        link_mismatches += 1

source_paths_present = sum(Path(row['events_path']).is_file() for row in trajectories if isinstance(row.get('events_path'), str) and row['events_path'])
event_root = os.environ.get('SWE_GYM_PLUS_EVENT_ROOT')
mirrored_events_present = None
if event_root:
    event_files = list(Path(event_root).glob('*/events.jsonl'))
    event_parent_ids = {path.parent.name for path in event_files}
    mirrored_events_present = sum(row.get('run_id') in event_parent_ids for row in trajectories)
historical_audit_path = DATA_ROOT / 'logs' / 'audit.log'
historical_audit = json.loads(historical_audit_path.read_text(encoding='utf-8').splitlines()[0]) if historical_audit_path.is_file() else {}
historical_sources = historical_audit.get('sources', [])
historical_source_tasks = {json.dumps(row.get('task_id'), sort_keys=True) for row in historical_sources}
historical_event_hashes = [row.get('events_sha256') for row in historical_sources]
hashes_complete = all(isinstance(value, str) and len(value) == 64 and all(ch in '0123456789abcdef' for ch in value.lower()) for value in historical_event_hashes)

print({'trajectory_tasks_without_actions': len(trajectory_task_ids - action_task_ids),
       'action_tasks_without_trajectory': len(action_task_ids - trajectory_task_ids),
       'duplicate_trajectory_task_rows': duplicate_trajectory_tasks,
       'action_to_trajectory_link_mismatches': link_mismatches,
       'original_event_paths_available': source_paths_present, 'trajectory_count': len(trajectories),
       'mirrored_event_receipts_available': mirrored_events_present,
       'historical_audit_claims_real_receipts_verified': historical_audit.get('real_receipts_verified'),
       'historical_event_hashes': len(historical_event_hashes), 'historical_event_hashes_well_formed': hashes_complete,
       'historical_audit_task_coverage': len(historical_source_tasks & trajectory_task_ids)})

In [ ]:
# Check the immutable task split when its private path is supplied.
split_path = os.environ.get('SWE_GYM_PLUS_SPLIT_FILE')
if split_path:
    split_path = Path(split_path)
    split = json.loads(split_path.read_text(encoding='utf-8'))
    split_sets = {name: {json.dumps(task, sort_keys=True) for task in split.get(name, [])}
                  for name in ('train', 'dev', 'eval', 'excluded_prior_exposure')}
    intersections = {f'{left}_vs_{right}': len(split_sets[left] & split_sets[right])
                     for i, left in enumerate(split_sets) for right in list(split_sets)[i+1:]}
    experiment_path = DATA_ROOT / 'experiment.json'
    expected_hash = json.loads(experiment_path.read_text(encoding='utf-8')).get('split_sha256') if experiment_path.is_file() else None
    actual_hash = hashlib.sha256(split_path.read_bytes()).hexdigest()
    print({'split_counts': {name: len(values) for name, values in split_sets.items()},
           'action_tasks_outside_train': len(action_task_ids - split_sets['train']),
           'action_tasks_overlapping_dev': len(action_task_ids & split_sets['dev']),
           'action_tasks_overlapping_eval': len(action_task_ids & split_sets['eval']),
           'action_tasks_overlapping_prior_exposure': len(action_task_ids & split_sets['excluded_prior_exposure']),
           'split_partition_intersections': intersections,
           'split_sha256_matches_experiment': expected_hash == actual_hash})
else:
    print('Split check not run: set SWE_GYM_PLUS_SPLIT_FILE to the frozen private split file.')

In [ ]:
# Optional exact chat-template target length check against the inference cap.
tokenizer_path = os.environ.get('SWE_GYM_PLUS_TOKENIZER')
if tokenizer_path:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, local_files_only=True)
    target_lengths, prefix_failures = [], 0
    for row in actions:
        messages = row.get('messages') or []
        if not messages or messages[-1].get('role') != 'assistant':
            prefix_failures += 1
            continue
        prefix_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
        full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        prefix_ids = tokenizer(prefix_text, add_special_tokens=False)['input_ids']
        full_ids = tokenizer(full_text, add_special_tokens=False)['input_ids']
        if full_ids[:len(prefix_ids)] != prefix_ids:
            prefix_failures += 1
        target_lengths.append(len(full_ids) - len(prefix_ids))
    generation_cap = int(os.environ.get('SWE_GYM_PLUS_MAX_NEW_TOKENS', '768'))
    print({'chat_template_target_max': max(target_lengths, default=0),
           'chat_template_prefix_stability_failures': prefix_failures,
           'targets_above_inference_generation_cap': sum(n > generation_cap for n in target_lengths),
           'inference_generation_cap': generation_cap})
else:
    print('Tokenizer check not run: set SWE_GYM_PLUS_TOKENIZER to a local tokenizer directory.')